In [1]:
import torch
# try:
#     import torch_xla
#     import torch_xla.core.xla_model as xm
#     import torch_xla.distributed.parallel_loader as pl
#     TPU_AVAILABLE = True
# except ImportError:
#     TPU_AVAILABLE = False
    
# if TPU_AVAILABLE:
#     DEVICE = torch_xla.device()
#     !pip install soundfile -q
#     !pip install torchinfo -q
# else:
    #DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TO-DO:
# - Spectrogram augmentation/freqmasking/timemasking
# - Temperature scaling
# - combine standard attn pooling with MS pooling
!python --version
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"USING DEVICE {DEVICE}")

Python 3.12.12
USING DEVICE cuda


In [2]:
import os
import sys
import random
import math
import ast
import warnings
from pathlib import Path

# Audio processing modules
import soundfile as sf

# Data handling and visualization modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Skikit-learn preprocessing modules
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import KFold
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import StratifiedGroupKFold

# PyTorch modules
import torch
#torch.autograd.set_detect_anomaly(True)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
import torchvision.models as models
import torchvision.transforms as TV
from torchinfo import summary

#/kaggle/input/competitions/birdclef-2026/

In [3]:
EXAMPLE_AUDIO_PATH = "/kaggle/input/competitions/birdclef-2026/train_audio/1161364/iNat1114648.ogg"

SAMPLE_RATE = 32000

#transform params
N_MELS = 64
N_FFT = 512
HOP_LENGTH = 320
SAMPLE_DURATION = 5

N_SAMPLES = SAMPLE_RATE * SAMPLE_DURATION   # Number of samples of preprocessed audio files
F_MIN       = 20                            # Minimum frequency
F_MAX       = 16000                         # Maximum frequency
BATCH_SIZE = 128

WARM_START = True
MASKING = False
SUBMITTING = False

# SESSION = ort.InferenceSession(
#     "/kaggle/input/models/fluidize/perch-v2-onnx/onnx/default/1/perch_v2.onnx",
#     providers=["CUDAExecutionProvider"]
# )

In [4]:
def generate_audio_slices(audio_full_paths):
    slices =[]
    # i = 0
    for audio_full_path in tqdm(audio_full_paths):
        info = sf.info(audio_full_path)
        total_samples = info.frames
    
        start_frame = 0
        end_frame = N_SAMPLES
        
        # if i == 10:
        #     break
    
        while start_frame < total_samples: #make sure the next frame doesnt go over
            #replace .ogg with required submission id
            row_id = os.path.basename(audio_full_path).replace(".ogg", f"_{int(end_frame/SAMPLE_RATE)}") 
           
            slices.append((row_id, audio_full_path, start_frame, end_frame))
            start_frame += N_SAMPLES
            end_frame += N_SAMPLES

        # i += 1

    return pd.DataFrame(slices, columns=["row_id", "full_path", "start_frame", "end_frame"])

In [5]:
if not SUBMITTING:
    if os.path.exists("/kaggle/input/datasets/fluidize/birdclef-processed/processed_trainval.csv"):
        trainval = pd.read_csv("/kaggle/input/datasets/fluidize/birdclef-processed/processed_trainval.csv")
    else:
        trainval = pd.read_csv('/kaggle/input/competitions/birdclef-2026/train.csv')
        #conv trainval start/end sec -> frames
        trainval_extra = pd.read_csv('/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv')
        trainval_extra["extra_start_frame"] = pd.to_timedelta(trainval_extra['start']).dt.total_seconds().astype(int) * SAMPLE_RATE
        trainval_extra["extra_end_frame"] = pd.to_timedelta(trainval_extra['end']).dt.total_seconds().astype(int) * SAMPLE_RATE
        trainval_extra.drop(columns=["start","end"], inplace=True)
        #print(trainval_extra.head())
        taxonomy = pd.read_csv('/kaggle/input/competitions/birdclef-2026/taxonomy.csv')
        
        # Merge and standardize label informations from different columns of data source tables (all labels in a single string seperated by ;)
        trainval['all_labels'] = trainval.apply(lambda row: ';'.join([row['primary_label']] + ast.literal_eval(row['secondary_labels'])), axis=1)
        trainval_extra = trainval_extra.drop_duplicates().reset_index(drop=True).rename(columns={'primary_label': 'all_labels'})
        trainval_extra['primary_label'] = trainval_extra['all_labels'].str.split(';').str[0]
        trainval_extra['secondary_labels'] = trainval_extra['all_labels'].str.split(';').str[1:]
        
        # Include full paths in the data source tables
        trainval['full_path'] = '/kaggle/input/competitions/birdclef-2026/train_audio/' + trainval['filename']
        trainval_extra['full_path'] = '/kaggle/input/competitions/birdclef-2026/train_soundscapes/' + trainval_extra['filename']
        
        # Create a list of 5 sec slices of audio files and merge information into trainval
        audio_full_paths = trainval['full_path'].unique().tolist()
        slices_df = generate_audio_slices(audio_full_paths)
        trainval = pd.merge(trainval, slices_df, on='full_path', how='right')
        
        # Merge the two datasets into one single dataframe
        trainval = pd.concat([trainval, trainval_extra], axis=0)
        trainval["start_frame"] = trainval["start_frame"].combine_first(trainval["extra_start_frame"])
        trainval["end_frame"] = trainval["end_frame"].combine_first(trainval["extra_end_frame"])
        trainval.drop(columns=["extra_start_frame","extra_end_frame"], inplace=True)
        
        trainval.to_csv("processed_trainval.csv")
    
    #trainval = trainval.iloc[0:10000]
    print(trainval.head())

   Unnamed: 0 primary_label secondary_labels type  latitude  longitude  \
0           0       1161364               []   []  -22.7562   -46.8666   
1           1       1161364               []   []  -22.7562   -46.8666   
2           2       1161364               []   []  -22.7562   -46.8666   
3           3       1161364               []   []  -22.7558   -46.8700   
4           4       1161364               []   []  -22.7558   -46.8700   

  scientific_name   common_name class_name  inat_taxon_id  ...   license  \
0    Guyalna cuta  Guyalna cuta    Insecta      1161364.0  ...  cc-by-nc   
1    Guyalna cuta  Guyalna cuta    Insecta      1161364.0  ...  cc-by-nc   
2    Guyalna cuta  Guyalna cuta    Insecta      1161364.0  ...  cc-by-nc   
3    Guyalna cuta  Guyalna cuta    Insecta      1161364.0  ...  cc-by-nc   
4    Guyalna cuta  Guyalna cuta    Insecta      1161364.0  ...  cc-by-nc   

  rating                                                url  \
0    0.0  https://static.inaturalis

In [6]:
#one hot encoder
if not SUBMITTING:
    all_unique_labels = trainval['all_labels'].str.split(';').explode().unique()
    mlb = MultiLabelBinarizer()
    mlb.fit([all_unique_labels])
    
    label_matrix = mlb.transform(trainval['all_labels'].str.split(';'))
    label_counts = np.sum(label_matrix, axis=0)
    
    label_weights = (1 / label_counts) / np.min(1 / label_counts)
    label_weights = np.clip(label_weights, None, 50) #clip to prevent overweighting
    #print(label_weights)
    print(f"Total unique labels: {len(all_unique_labels)}")
    #print(f"Label classes: {mlb.classes_}")

Total unique labels: 234


In [7]:
class MultiLabelLoss(nn.Module):
    def __init__(self, device, pos_weight):
        super().__init__()
        self.pos_weight = torch.Tensor(pos_weight).to(device)

    def forward(self, logits, targets):
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, weight=None, pos_weight=self.pos_weight, reduction='mean') #sigmoid already done internally
        return bce_loss

In [8]:
class AnimalSoundDataset(Dataset):
    def __init__(self, dataframe, mlb=None, submitting=False):
        self.dataframe = dataframe
        self.mlb = mlb
        self.sample_rate = SAMPLE_RATE
        self.max_audio_samples = N_SAMPLES
        self.submitting = submitting
        
    def __len__(self):
        return len(self.dataframe)

    def _load_audio_slice_at(self, path, start_sample, end_sample):
        stereo_waveform, sr = sf.read(path, start=start_sample.astype(int), stop=end_sample.astype(int), dtype='float32', always_2d=True)
        waveform = stereo_waveform.mean(axis=1)
        waveform = torch.from_numpy(waveform).to(torch.float32).contiguous()
        waveform = waveform.unsqueeze(0)
        return waveform, sr

    def _process_audio(self, audio_path_full, start_samples, end_samples):
        # Cut 5 sec audio sample with defined starting time from given audio file
        waveform, sr = self._load_audio_slice_at(audio_path_full, start_samples, end_samples)

        # Resample waveform if necessary
        if sr != self.sample_rate:
            resampler = T.Resample(orig_freq=sr, new_freq=self.sample_rate)
            waveform = resampler(waveform)

        # Padding or truncating sample if necessary
        current_n_samples = waveform.shape[1]
        if current_n_samples < self.max_audio_samples:
            padding_needed = self.max_audio_samples - current_n_samples
            waveform = F.pad(waveform, (0, padding_needed))
        elif current_n_samples > self.max_audio_samples:
            waveform = waveform[:, :self.max_audio_samples]
            
        return waveform

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        row_id = row['row_id']
        audio_path_full = row['full_path']
        start_samples = row['start_frame']
        end_samples = row['end_frame']
        waveform = self._process_audio(audio_path_full, start_samples, end_samples)
        if not self.submitting:
            labels = row['all_labels'].split(';')
            one_hot_labels = torch.tensor(self.mlb.transform([labels]).squeeze(0), dtype=torch.float32)
            return (waveform, one_hot_labels)
        else:
            return (waveform, row_id)

In [9]:
class MelSpecHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel_transform = T.MelSpectrogram(
            sample_rate=SAMPLE_RATE,
            n_fft=N_FFT,        # FFT window size
            hop_length=HOP_LENGTH,    # step size between frames
            n_mels=N_MELS,        # number of mel bands
            f_min=F_MIN,
            f_max=F_MAX,
            power=2.0          # power spectrogram (2.0 = energy)
        )

        self.mfcc_transform = T.MFCC(
            sample_rate=SAMPLE_RATE,
            n_mfcc=N_MELS, # Set this to N_MELS to keep height consistent
            melkwargs={
                "n_fft": N_FFT,
                "hop_length": HOP_LENGTH,
                "n_mels": N_MELS,
                "f_min": F_MIN,
                "f_max": F_MAX,
                "power": 2.0,
            }
        )

        self.lfcc_transform = T.LFCC(
            sample_rate=SAMPLE_RATE,
            n_lfcc=N_MELS, # Set this to N_MELS to keep height consistent
            speckwargs={
                "n_fft": N_FFT,
                "hop_length": HOP_LENGTH,
                "power": 2.0,
            }
        )
        
        self.amp_to_db_transform = T.AmplitudeToDB(top_db=80)

        self.spec_freq_mask = T.FrequencyMasking(freq_mask_param=10, iid_masks=True)
        self.cep_freq_mask = T.FrequencyMasking(freq_mask_param=5, iid_masks=True)
    
    def forward(self, waveform):
        mel_S = self.mel_transform(waveform)
        mel_S = torch.clamp(mel_S, min=1e-10)          # guard log(0)
        mel_S_db = self.amp_to_db_transform(mel_S)
        mel_S_db = torch.nan_to_num(mel_S_db, nan=0.0, posinf=0.0, neginf=0.0)  # guard any surviving infs/nans
        smp_min = mel_S_db.amin(dim=(1,2), keepdim=True)
        smp_max = mel_S_db.amax(dim=(1,2), keepdim=True)
        mel_S_db_norm = (mel_S_db - smp_min) / (smp_max - smp_min + 1e-8)

        mfcc = self.mfcc_transform(waveform)
        lfcc = self.lfcc_transform(waveform)

        if (self.training and MASKING)
            batch_size, channels, freq, time_len = mel_S_db_norm.shape
            for _ in range(2):
                t_width = random.randint(1, 45) 
                

                t_starts = [random.randint(0, time_len - t_width) for _ in range(batch_size)]
                

                for i, t_start in enumerate(t_starts):
                    mel_S_db_norm[i, :, :, t_start:t_start+t_width] = 0.0
                    mfcc[i, :, :, t_start:t_start+t_width] = 0.0
                    lfcc[i, :, :, t_start:t_start+t_width] = 0.0


            for _ in range(2):
                mel_S_db_norm = self.spec_freq_mask(mel_S_db_norm)
                
            mfcc = self.cep_freq_mask(mfcc)
            lfcc = self.cep_freq_mask(lfcc)
        
        return mel_S_db_norm, mfcc, lfcc

class ImageGLU(nn.Module):
    def __init__(self):
        super().__init__()
        self.map_generator = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, padding=1),
            nn.BatchNorm2d(1),
            nn.GELU(),
            nn.Conv2d(in_channels=1, out_channels=1, kernel_size=5, padding=2),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

    def forward(self, x):
        weight_map = self.map_generator(x)
        return weight_map * x

class MultiscalePooling(nn.Module):
    def __init__(self, reduction_dim=128):
        super().__init__()

        self.reduction_dim = reduction_dim
        
        self.aap2d_1 = nn.AdaptiveAvgPool2d((1,1))
        self.aap2d_2 = nn.AdaptiveAvgPool2d((1,4))
        self.aap2d_3 = nn.AdaptiveAvgPool2d((1,8))

        self.channel_reducer = nn.Conv2d(1280, self.reduction_dim, 1)

        self.output_dim = self.reduction_dim*(1+4+8)

    def forward(self, x):
        x1 = self.aap2d_1(x)
        x1 = self.channel_reducer(x1).flatten(1) #(B,64)

        x2 = self.aap2d_2(x)
        x2 = self.channel_reducer(x2).flatten(1) #(B,256)

        x3 = self.aap2d_3(x)
        x3 = self.channel_reducer(x3).flatten(1) #(B,512)

        x = torch.cat([x1,x2,x3], dim=1)

        return x

class AttentionPooling(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1280, 1, 1)
        self.output_dim = 1280
    
    def forward(self, x):
        logits = self.conv(x)
        weights = torch.sigmoid(logits) #does not sum to one

        return (x*weights).sum(dim=[2,3]) #outputs (B, 1280)

class EfficientNetHead(nn.Module):
    def __init__(self):
        super().__init__()

        #get raw convolution
        weights = "IMAGENET1K_V1" if not SUBMITTING else None #boost training
        print(f"Using weights {weights}")
        effnet = models.efficientnet_b0(weights=weights)
        effnet_modules = list(effnet.children())[:-2] #remove stock output fc + aap2d
        self.effnet = nn.Sequential(*effnet_modules)

        self.msp = MultiscalePooling(128)
        self.ap = AttentionPooling()
        self.output_dim = self.msp.output_dim + self.ap.output_dim

    def forward(self, x):
        raw_image = self.effnet(x)

        x1 = self.msp(raw_image)
        x2 = self.ap(raw_image)

        x = torch.concat([x1,x2], dim=1)
        
        return x

class VectorGate(nn.Module):
    def __init__(self, in_dim, compressed_dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(in_dim, compressed_dim),
            nn.GELU(),
            nn.Linear(compressed_dim, in_dim),
            nn.Sigmoid()
        )

        nn.init.xavier_uniform_(self.gate[0].weight)
        nn.init.zeros_(self.gate[0].bias)

        nn.init.xavier_uniform_(self.gate[2].weight)
        nn.init.constant_(self.gate[2].bias, 5.0) #sigmoid will output ~1, start off as identity layer to prevent embedding destruction
    
    def forward(self, x):
        weights = self.gate(x)
        return weights * x

class Classifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.spectrogram_head = MelSpecHead()
        self.image_glu = ImageGLU()
        self.embedding_head = EfficientNetHead()
        self.embedding_head_output_dim = self.embedding_head.output_dim
        self.vector_gate = VectorGate(in_dim=self.embedding_head_output_dim, compressed_dim=32)
        
        self.out = nn.Sequential(
            self.vector_gate,
            nn.Linear(self.embedding_head_output_dim,512),
            nn.GELU(),
            nn.Linear(512,num_classes)
        )
        
    def forward(self, waveform):
        spec, mfcc, lfcc = self.spectrogram_head(waveform)
        focused_spec = self.image_glu(spec)
        focused_spec = spec
        combined_input = torch.cat([focused_spec, mfcc, lfcc], dim=1)

        embedding = self.embedding_head(combined_input)
        
        out = self.out(embedding)
        return out

In [10]:
summary(Classifier(234))

Using weights IMAGENET1K_V1
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 144MB/s] 


Layer (type:depth-idx)                                            Param #
Classifier                                                        --
├─MelSpecHead: 1-1                                                --
│    └─MelSpectrogram: 2-1                                        --
│    │    └─Spectrogram: 3-1                                      --
│    │    └─MelScale: 3-2                                         --
│    └─MFCC: 2-2                                                  --
│    │    └─AmplitudeToDB: 3-3                                    --
│    │    └─MelSpectrogram: 3-4                                   --
│    └─LFCC: 2-3                                                  --
│    │    └─AmplitudeToDB: 3-5                                    --
│    │    └─Spectrogram: 3-6                                      --
│    └─AmplitudeToDB: 2-4                                         --
│    └─FrequencyMasking: 2-5                                      --
│    └─FrequencyMasking: 2-6 

In [11]:
if not SUBMITTING:
    NUM_CLASSES = len(mlb.classes_)
    NUM_EPOCHS = 5
    NUM_WORKERS = 4

    model = Classifier(NUM_CLASSES).to(DEVICE)
    LEARNING_RATE = 1e-4
    LEARNING_CONFIG = [
        {"params" : model.embedding_head.effnet.parameters(), "lr" : 1e-5}
    ]
    
    def get_metrics(logits_np, targets_np):
        present_classes_mask = np.any(targets_np == 1, axis=0)
        filtered_logits = logits_np[:, present_classes_mask]
        filtered_targets = targets_np[:, present_classes_mask]
        probs = 1.0 / (1.0 + np.exp(-filtered_logits))
        map_score = average_precision_score(filtered_targets, probs, average="macro")
        roc_score = roc_auc_score(filtered_targets, probs, average="macro")
        return map_score, roc_score
    
    def train_one_epoch(model, loader, criterion, optimizer, device):
        model.train()
        running = 0.0
        n = 0
        
        for waveforms, targets in tqdm(loader, desc="train", leave=False):
            waveforms = waveforms.to(device)
            targets = targets.to(device)
            optimizer.zero_grad(set_to_none=True)
            
            logits = model(waveforms)
            loss = criterion(logits, targets)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            bs = waveforms.size(0)
            running += loss.item() * bs
            n += bs
            
        return running / max(n, 1)
    
    @torch.no_grad()
    def evaluate(model, loader, criterion, device):
        model.eval()
        running = 0.0
        n = 0
        all_logits = []
        all_targets = []
        
        for waveforms, targets in tqdm(loader, desc="val", leave=False):
            waveforms = waveforms.to(device)
            targets = targets.to(device)
            logits = model(waveforms)
            loss = criterion(logits, targets)
            bs = waveforms.size(0)
            running += loss.item() * bs
            n += bs
            all_logits.append(logits.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
            
        logits_np = np.concatenate(all_logits, axis=0)
        targets_np = np.concatenate(all_targets, axis=0)

        map_score, roc_score = get_metrics(logits_np, targets_np)
        return running / max(n, 1), map_score, roc_score
    
    train_df = trainval.sample(frac=0.9, random_state=42)
    val_df = trainval.drop(train_df.index)
    print(f"Train Size: {len(train_df)} Val Size: {len(val_df)}")
    
    train_loader = DataLoader(
        AnimalSoundDataset(train_df.reset_index(drop=True), mlb=mlb, submitting=False),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        persistent_workers=True,
        pin_memory=DEVICE == "cuda",
        prefetch_factor=2
    )
    val_loader = DataLoader(
        AnimalSoundDataset(val_df.reset_index(drop=True), mlb=mlb, submitting=False),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        persistent_workers=True,
        pin_memory=DEVICE == "cuda",
        prefetch_factor=2
    )
    
    #model = torch.compile(model)
    criterion = MultiLabelLoss(device=DEVICE, pos_weight=label_weights)
    optimizer = torch.optim.AdamW(LEARNING_CONFIG, lr=LEARNING_RATE)
    
    if WARM_START:
        try:
            checkpoint = torch.load("/kaggle/working/checkpoint.pth", map_location=DEVICE)
            model.load_state_dict(checkpoint.get("model"))
            optimizer.load_state_dict(checkpoint.get("optimizer"))
            print("WARM START SUCCESSFUL")
        except Exception as e:
            print(f"WARM START WARNING: {e}")
    
    train_losses = []
    val_losses = []
    val_map_scores = []
    val_roc_scores = []
    best_val_loss = float('inf')
    for epoch in range(NUM_EPOCHS):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_map, val_roc = evaluate(model, val_loader, criterion, DEVICE)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_map_scores.append(val_map)
        val_roc_scores.append(val_roc)
        print(f"epoch {epoch + 1}/{NUM_EPOCHS} train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_mAP={val_map:.4f} val_roc={val_roc:.4f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
            }, "checkpoint.pth")

    #tempscaling
    @torch.no_grad()
    def collect_logits(model, loader, device):
        model.eval()
        all_logits, all_targets = [], []
        for waveforms, targets in tqdm(loader, desc="Temp Scaling", leave=False):
            waveforms = waveforms.to(device)
            logits = model(waveforms)
            all_logits.append(logits.cpu())
            all_targets.append(targets)
        return torch.cat(all_logits), torch.cat(all_targets)

    def temperature_nll(temperature, logits, targets):
        scaled = logits / temperature
        loss = torch.nn.functional.binary_cross_entropy_with_logits(scaled, targets)
        return loss.item()

    val_logits, val_targets = collect_logits(model, val_loader, DEVICE)

    from scipy.optimize import minimize_scalar
    result = minimize_scalar(
        temperature_nll,
        bounds=(0.01, 10.0),
        method="bounded",
        args=(val_logits, val_targets.float())
    )
    temperature = float(result.x)
    print(f"optimal temperature: {temperature:.4f}")

    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "temperature": temperature,
    }, f"effnetb1-{val_map_scores[-1]}.pth")
    
    fig, axes = plt.subplots(1,2)
    axes[0].plot(train_losses, marker='o')
    axes[0].plot(val_losses, marker='o')
    axes[1].plot(val_map_scores, marker='o')
    axes[1].plot(val_roc_scores, marker='o')
    plt.show()

Using weights IMAGENET1K_V1
Train Size: 208003 Val Size: 23111
WARM START WARNING: [Errno 2] No such file or directory: '/kaggle/working/checkpoint.pth'


epoch 1/5 train_loss=0.3981 val_loss=0.2497 val_mAP=0.0092 val_roc=0.5828


KeyboardInterrupt: 

In [ ]:
if SUBMITTING:
    OUT_COLUMNS = out_columns = [
    'row_id', '1161364', '116570', '1176823', '1491113', '1595929', '209233', '22930',
    '22956', '22961', '22967', '22973', '22983', '22985', '23150', '23154',
    '23158', '23176', '23724', '24279', '24285', '24287', '24321', '244024',
    '25073', '25092', '25214', '326272', '41970', '43435', '47144', '47158son01',
    '47158son02', '47158son03', '47158son04', '47158son05', '47158son06', '47158son07', '47158son08',
    '47158son09', '47158son10', '47158son11', '47158son12', '47158son13', '47158son14', '47158son15', '47158son16',
    '47158son17', '47158son18', '47158son19', '47158son20', '47158son21', '47158son22', '47158son23', '47158son24',
    '47158son25', '476521', '516975', '517063', '555123', '555145', '555146', '64898',
    '65377', '65380', '66971', '67107', '67252', '70711', '738183', '74113',
    '74580', '760266', 'ashgre1', 'astcra1', 'bafcur1', 'baffal1', 'banana', 'barant1',
    'batbel1', 'baymac', 'bbwduc', 'bcwfin2', 'bkcdon', 'bkhpar', 'blchaw1', 'blheag1',
    'blttit1', 'bncfly', 'bobfly1', 'brcmar1', 'brnowl', 'bucmot4', 'bucpar', 'bufpar',
    'bunibi1', 'burowl', 'camfli1', 'chacha1', 'chbmoc1', 'chobla1', 'chvcon1', 'cibspi1',
    'coffal1', 'compau', 'compot1', 'crbthr1', 'crebec1', 'dwatin1', 'epaori4', 'eulfly1',
    'fabwre1', 'fepowl', 'ficman1', 'flawar1', 'fotfly', 'fusfly1', 'gilhum1', 'giwrai1',
    'glteme1', 'grasal3', 'greani1', 'greant1', 'greela', 'grekis', 'grepot1', 'gretho2',
    'greyel', 'grfdov1', 'grhtan1', 'gycwor1', 'horscr1', 'houspa', 'hyamac1', 'larela1',
    'lesela1', 'lesgrf1', 'limpki', 'linwoo1', 'litcuc2', 'litnig1', 'mabpar', 'magant1',
    'magtan2', 'masgna1', 'nacnig1', 'ocecra1', 'oliwoo1', 'orbtro3', 'orwpar', 'osprey',
    'pabspi1', 'palhor3', 'paltan1', 'phecuc1', 'picpig2', 'pirfly1', 'plasla1', 'platyr1',
    'plcjay1', 'pluibi1', 'purjay1', 'pvttyr1', 'ragmac1', 'rebscy1', 'recfin1', 'redjun',
    'relser1', 'rinkin1', 'rivwar1', 'roahaw', 'rubthr1', 'rufcac2', 'rufcas2', 'rufgna3',
    'rufhor2', 'rufnig1', 'ruftho1', 'ruftof1', 'rumfly1', 'ruther1', 'rutjac1', 'sabspa1',
    'saffin', 'saytan1', 'scadov1', 'schpar1', 'scther1', 'shcfly1', 'shshaw', 'shtnig1',
    'sibtan2', 'smbani', 'smbtin1', 'sobcac1', 'sobtyr1', 'socfly1', 'sofspi1', 'souant1',
    'soulap1', 'souscr1', 'spbant3', 'spispi1', 'sptnig1', 'squcuc1', 'stbwoo2', 'strcuc1',
    'strher2', 'strowl1', 'swthum1', 'swtman1', 'tattin1', 'thlwre1', 'toctou1', 'trokin',
    'trsowl', 'undtin1', 'varant1', 'watjac1', 'wesfie1', 'wfwduc1', 'whbant2', 'whbwar2',
    'whiwoo1', 'whlspi1', 'whnjay1', 'whtdov', 'whwpic1', 'y00678', 'yebcar', 'yebela1',
    'yecmac', 'yecpar', 'yehcar1', 'yeofly1',
]

In [ ]:
if SUBMITTING:
    model = Classifier(234)
    checkpoint = torch.load("/kaggle/input/models/fluidize/birdclef-effnetb1/pytorch/multiscalepooling/1/effnetb1-0.8404588967180346.pth", map_location=DEVICE)
    model.load_state_dict(checkpoint.get("model"), strict=True)
    model.to(DEVICE)
    temperature = float(checkpoint.get("temperature", 1.0))
    print(f"loaded temperature: {temperature:.4f}")
    
    
    test_path = "/kaggle/input/competitions/birdclef-2026/test_soundscapes"
    #test_path = "/kaggle/input/competitions/birdclef-2026/train_soundscapes"
    with os.scandir(test_path) as audio_files:
        test_audio_full_paths = [os.path.abspath(file.path) for file in audio_files if file.name.endswith(".ogg")]
        testset = generate_audio_slices(test_audio_full_paths)
        test_loader = DataLoader(
            AnimalSoundDataset(testset.reset_index(drop=True), submitting=True),
            batch_size=64,
            shuffle=False,
            num_workers=4,
            pin_memory=DEVICE == "cuda",
            prefetch_factor=2
        )
    
    test_results = []
    
    with torch.no_grad():
        model.eval()
        for waveforms, row_ids in tqdm(test_loader):
            waveforms = waveforms.to(DEVICE)
            logits = F.sigmoid(model(waveforms) / temperature)
            logits = logits.detach().cpu().numpy()

            for i, row_id in enumerate(row_ids): #unbatch
                test_row = [row_id, *logits[i]]
                test_results.append(test_row)
        
    OUTPUT = pd.DataFrame(test_results, columns=OUT_COLUMNS)
    print(OUTPUT)
    # print(out_columns)
    OUTPUT.to_csv("submission.csv", index=False)
            
        